# 04 - Full-scale Ablation, Controls and Statistical Tests
**Addresses comments 1 and 3 (and 4 for HP-Err / edge metrics).**

Everything is read from the multi-seed runs of notebook 03 - **same protocol, same epochs, same split** for every row (the submitted ablation used 200 images, 3 epochs and no seeds).

Two levels of statistics, both reported:
* **seed level** - mean +- std over independent training runs;
* **image level** - paired comparison on the *same test images* (seed-averaged per-image metric): paired t, Wilcoxon, paired-bootstrap 95% CI, Cohen's d_z, **Holm-corrected** across all comparisons.

Holm correction is applied within each (metric, reference) family of comparisons. Claims are evaluated by a **pre-registered rule** (below) and the notebook prints SUPPORTED / NOT SUPPORTED - the manuscript must follow it.

In [1]:
import os, sys, json, warnings, importlib
PROJECT_ROOT = os.path.abspath("..")
if not os.path.isdir(os.path.join(PROJECT_ROOT, "notebooks")):
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
# Prefer a configured data root only when it actually contains the paired folders.
def _has_paired_data(root):
    if not root:
        return False
    return ((os.path.isdir(os.path.join(root, "train", "train", "GT")) and
             os.path.isdir(os.path.join(root, "train", "train", "NoisyLR"))) or
            (os.path.isdir(os.path.join(root, "GT")) and
             os.path.isdir(os.path.join(root, "NoisyLR"))))
if not _has_paired_data(os.environ.get("SEMICON_DATA")):
    os.environ["SEMICON_DATA"] = PROJECT_ROOT
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import config as C
C = importlib.reload(C)
# Print explicit results and data roots below
from utils.experiment import load_results, load_per_image, ABLATION, LADDER
from utils import stats as S, figures as FG
RUNS = os.path.join(PROJECT_ROOT, "results_kaggle")
DATA = PROJECT_ROOT
RUNS_DIR = os.path.join(RUNS, "runs")
TABLES_DIR = os.path.join(RUNS, "tables")
FIGS_DIR = os.path.join(RUNS, "figures")
print("RUNS =", RUNS)
print("DATA =", DATA)
if not os.path.isdir(os.path.join(DATA, "train", "train", "GT")) or not os.path.isdir(os.path.join(DATA, "train", "train", "NoisyLR")):
    raise FileNotFoundError(f"Paired training data not found under DATA={DATA}")
res = load_results(RUNS_DIR)
if res.empty:
    raise FileNotFoundError(f"No completed training results found under RUNS={RUNS_DIR}. Check runs/<model>/seed*/results.json.")
have = [c for c in ABLATION if c in set(res.config)]
if not have:
    raise ValueError(f"No ablation results found under RUNS={RUNS_DIR}; available columns: {list(res.columns)}")
print("configs found:", have); print(res.groupby("config").seed.apply(list).to_string())

RUNS = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_kaggle
DATA = d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow
configs found: ['baseline', 'global_gc', 'film_control', 'full']
config
NAFNet-small       [0, 1, 2, 3]
Restormer-small    [0, 1, 2, 3]
SwinIR-small       [0, 1, 2, 3]
baseline           [0, 1, 2, 3]
film_control       [0, 1, 2, 3]
full               [0, 1, 2, 3]
global_gc                [0, 1]


## 1. Main table: mean +- std over seeds (test split)

In [2]:
metrics = [("test_psnr", "PSNR", True), ("test_ssim", "SSIM", True), ("test_hp_err", "HP-Err (top-5%)", False),
           ("test_edge_corr", "Edge corr", True), ("test_edge_f1", "Edge F1", True), ("test_flat_err", "Flat-region err", False)]
blocks = {}
for col, nm, hb in metrics:
    blocks[nm] = S.summarize_configs(res, col, have).set_index("config")
tab = pd.DataFrame({nm: [f"{b.loc[c,'mean']:.4f} +- {b.loc[c,'std']:.4f}" for c in have] for nm, b in blocks.items()}, index=have)
tab.insert(0, "params", [int(res[res.config == c].params.iloc[0]) for c in have]); tab.insert(1, "n_seeds", [int((res.config == c).sum()) for c in have])
print(tab.to_string()); tab.to_csv(os.path.join(TABLES_DIR, "table_ablation_main.csv"))

              params  n_seeds               PSNR              SSIM   HP-Err (top-5%)         Edge corr           Edge F1   Flat-region err
baseline       86945        4  27.6459 +- 0.0553  0.7290 +- 0.0019  0.1074 +- 0.0008  0.8523 +- 0.0011  0.5966 +- 0.0016  0.0233 +- 0.0002
global_gc      86945        2  27.5955 +- 0.0491  0.7276 +- 0.0016  0.1080 +- 0.0007  0.8512 +- 0.0005  0.5954 +- 0.0011  0.0234 +- 0.0002
film_control  116138        4  27.6908 +- 0.0468  0.7295 +- 0.0013  0.1067 +- 0.0008  0.8535 +- 0.0011  0.5986 +- 0.0019  0.0233 +- 0.0002
full          116138        4  27.7690 +- 0.0278  0.7331 +- 0.0007  0.1057 +- 0.0004  0.8557 +- 0.0005  0.6011 +- 0.0009  0.0229 +- 0.0001


## 2. Paired image-level tests (Holm-corrected)
Reference rows: `baseline` (is anything better than the plain backbone?) and **`film_control`** (is the gain due to *distribution modelling* or merely to the extra encoder/FiLM capacity?).

In [3]:
per_image = {m: {c: load_per_image(RUNS_DIR, c, m) for c in have} for m in ["psnr", "ssim", "hp_err", "edge_corr", "flat_err"]}
tests = []
for m, hb in [("psnr", True), ("ssim", True), ("hp_err", False), ("edge_corr", True), ("flat_err", False)]:
    for ref in ["baseline", "film_control"]:
        if ref in have:
            t = S.compare_to_reference(per_image[m], ref, have, higher_better=hb); t["metric"] = m; tests.append(t)
tests = pd.concat(tests, ignore_index=True)
show = ["metric", "config", "vs", "mean_diff", "ci95_lo", "ci95_hi", "p_wilcoxon_holm", "cohens_dz", "frac_images_improved", "significant_0.05"]
print("(mean_diff > 0 always means 'config better than reference')")
print(tests[show].round(4).to_string(index=False)); tests.to_csv(os.path.join(TABLES_DIR, "table_ablation_paired_tests.csv"), index=False)

(mean_diff > 0 always means 'config better than reference')
   metric       config           vs  mean_diff  ci95_lo  ci95_hi  p_wilcoxon_holm  cohens_dz  frac_images_improved  significant_0.05
     psnr    global_gc     baseline    -0.0505  -0.0585  -0.0431           0.0000    -0.7176                0.1094              True
     psnr film_control     baseline     0.0449   0.0341   0.0566           0.0000     0.4376                0.7781              True
     psnr         full     baseline     0.1231   0.1102   0.1366           0.0000     1.0034                0.9531              True
     psnr     baseline film_control    -0.0449  -0.0566  -0.0341           0.0000    -0.4376                0.2219              True
     psnr    global_gc film_control    -0.0954  -0.1105  -0.0817           0.0000    -0.7277                0.1344              True
     psnr         full film_control     0.0781   0.0666   0.0898           0.0000     0.7428                0.8625              True
     ssim

## 3. Pre-registered claim checks
Supported only if `mean_diff > 0`, the paired-bootstrap CI excludes 0 **and** Holm-adjusted Wilcoxon p < 0.05.

In [4]:
def verdict(cfg, ref, metric="psnr"):
    r = tests[(tests.config == cfg) & (tests.vs == ref) & (tests.metric == metric)]
    if r.empty: return "n/a (config or reference not run)"
    r = r.iloc[0]; ok = r.mean_diff > 0 and r.ci95_lo > 0 and r.p_wilcoxon_holm < 0.05
    return f"{'SUPPORTED' if ok else 'NOT SUPPORTED'}  (diff={r.mean_diff:+.4f}, CI [{r.ci95_lo:+.4f},{r.ci95_hi:+.4f}], p_holm={r.p_wilcoxon_holm:.3g})"
claims = {
 "C1  Full > Baseline (PSNR)": verdict("full", "baseline"),
 "C2  Full > FiLM-control  (gain is due to distribution modelling, not capacity)": verdict("full", "film_control"),
 "C3  Full > Global-GC (PSNR)": verdict("full", "global_gc") if "global_gc" in have else "n/a",
 "C4  LDMH-loss-only differs from Baseline? (should be NOT SUPPORTED: backbone unaffected)": verdict("ldmh_loss_only", "baseline"),
 "C5  Full > Baseline (HP-Err, top-5% pixels; lower is better)": verdict("full", "baseline", "hp_err"),
}
for k, v in claims.items(): print(f"{k}\n      -> {v}")
means = {c: blocks['PSNR'].loc[c, 'mean'] for c in have}
order = sorted(means, key=means.get); print("\nobserved PSNR order (low->high):", " < ".join(order))
print("paper's 'monotonic Baseline < Global-GC < LDMH < Full' claim holds in means:", order.index("baseline") < order.index("global_gc") < order.index("ldmh_loss_only") < order.index("full") if all(c in order for c in ["baseline","global_gc","ldmh_loss_only","full"]) else "n/a")
json.dump(claims, open(os.path.join(TABLES_DIR, "claim_checks.json"), "w"), indent=1)

C1  Full > Baseline (PSNR)
      -> SUPPORTED  (diff=+0.1231, CI [+0.1102,+0.1366], p_holm=7.52e-51)
C2  Full > FiLM-control  (gain is due to distribution modelling, not capacity)
      -> SUPPORTED  (diff=+0.0781, CI [+0.0666,+0.0898], p_holm=1.47e-38)
C3  Full > Global-GC (PSNR)
      -> n/a (config or reference not run)
C4  LDMH-loss-only differs from Baseline? (should be NOT SUPPORTED: backbone unaffected)
      -> n/a (config or reference not run)
C5  Full > Baseline (HP-Err, top-5% pixels; lower is better)
      -> SUPPORTED  (diff=+0.0017, CI [+0.0015,+0.0019], p_holm=1.36e-42)

observed PSNR order (low->high): global_gc < baseline < film_control < full
paper's 'monotonic Baseline < Global-GC < LDMH < Full' claim holds in means: n/a


In [5]:
# effect vs. noise floor: seed-to-seed std of each config, and the baseline-vs-ldmh_loss_only gap (functionally identical models)
nf = {c: S.noise_floor(per_image["psnr"][c], None)["std_over_seeds"] for c in have}
print("seed-to-seed std of mean test PSNR:", {c: round(v, 4) for c, v in nf.items()})
if all(c in have for c in ["baseline", "ldmh_loss_only"]):
    g = abs(means["ldmh_loss_only"] - means["baseline"]); print(f"|ldmh_loss_only - baseline| = {g:.4f} dB  <- an upper bound on pure initialisation/seed effect for IDENTICAL functions")
if "full" in have and "baseline" in have:
    print(f"|full - baseline| = {abs(means['full']-means['baseline']):.4f} dB  vs. noise-floor std ~ {max(nf['baseline'], nf['full']):.4f} dB")

seed-to-seed std of mean test PSNR: {'baseline': 0.0553, 'global_gc': 0.0491, 'film_control': 0.0468, 'full': 0.0278}
|full - baseline| = 0.1231 dB  vs. noise-floor std ~ 0.0553 dB


## 4. Does the 'collapse fix' matter?  Component health per run
`full_clamp` reproduces the original parameterisation; `full_no_penalty` the loss used in the old Table V. A fix is only demonstrated if the clamp variant actually collapses *and* this changes restoration quality (it cannot change the forward pass directly: beta/alpha only enter the auxiliary NLL).

In [6]:
import glob
rows = []
for cfg in [c for c in ["full", "full_clamp", "full_no_penalty", "full_no_bal", "full_no_div", "ldmh_loss_only"] if c in have]:
    for p in sorted(glob.glob(os.path.join(RUNS_DIR, cfg, "seed*", "history.csv"))):
        h = pd.read_csv(p).iloc[-1]; b = [h.beta0, h.beta1, h.beta2]; u = [h.usage0, h.usage1, h.usage2]
        rows.append(dict(config=cfg, seed=int(p.split("seed")[-1].split(os.sep)[0]), betas=[round(x, 3) for x in b], usage=[round(x, 3) for x in u],
                         min_beta_dist=round(h.min_beta_dist, 3), any_beta_at_bound=bool(h.any_beta_at_bound), min_usage=round(min(u), 3),
                         all_betas_gt_1=all(x > 1 for x in b), heavy_tailed_component=any(x < 1 for x in b)))
ch = pd.DataFrame(rows); print(ch.to_string(index=False)); ch.to_csv(os.path.join(TABLES_DIR, "table_component_health.csv"), index=False)
if len(ch):
    print("\nAny learned component with beta < 1 (heavy-tailed, as Section IV motivates)?", bool(ch.heavy_tailed_component.any()))
    print("Any run with a beta pinned at a bound (sigmoid/clamp saturation)?", bool(ch.any_beta_at_bound.any()))

config  seed                 betas                 usage  min_beta_dist  any_beta_at_bound  min_usage  all_betas_gt_1  heavy_tailed_component
  full     0  [0.504, 1.554, 2.38] [0.194, 0.443, 0.363]          0.826              False      0.194           False                    True
  full     1 [0.503, 1.548, 2.385] [0.189, 0.444, 0.367]          0.837              False      0.189           False                    True
  full     2 [0.504, 1.558, 2.377] [0.193, 0.443, 0.364]          0.819              False      0.193           False                    True
  full     3 [0.504, 1.553, 2.386] [0.192, 0.446, 0.362]          0.833              False      0.192           False                    True

Any learned component with beta < 1 (heavy-tailed, as Section IV motivates)? True
Any run with a beta pinned at a bound (sigmoid/clamp saturation)? False


## 5. Figures and LaTeX

In [7]:
for col, nm, hb in metrics[:3]:
    s = S.summarize_configs(res, col, have)
    FG.bars_with_error(s, [col], f"{nm} (test, mean +- std over seeds)", FIGS_DIR, f"fig_ablation_{col}", ylabel=nm)
tex = S.latex_table(pd.DataFrame({"Config": have, **{nm: [blocks[nm].loc[c, 'mean'] for c in have] for nm in ["PSNR", "SSIM", "HP-Err (top-5%)"]}}),
                    ["Config", "PSNR", "SSIM", "HP-Err (top-5%)"], "{:.4f}", "Full-scale ablation (test split, mean over seeds).", "tab:ablation")
open(os.path.join(TABLES_DIR, "table_ablation.tex"), "w").write(tex); print(tex)

\begin{table}[t]
\centering\caption{Full-scale ablation (test split, mean over seeds).}\label{tab:ablation}
\begin{tabular}{lccc}
\hline
Config & PSNR & SSIM & HP-Err (top-5%) \\
\hline
baseline & 27.6459 & 0.7290 & 0.1074 \\
global_gc & 27.5955 & 0.7276 & 0.1080 \\
film_control & 27.6908 & 0.7295 & 0.1067 \\
full & 27.7690 & 0.7331 & 0.1057 \\
\hline
\end{tabular}
\end{table}
